<a name="top"></a><img src="../images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 3 插曲：Chisel 标准库
**上一节：[生成器：集合](3.2_collections.ipynb)**<br>
**下一节：[高阶函数](3.3_higher-order_functions.ipynb)**

## 动机
Chisel 的核心就是复用，因此提供一个接口标准库（鼓励 RTL 的互操作性）和常用硬件块的生成器是非常合理的。

## 环境设置

In [ ]:
val path = System.getProperty("user.dir") + "/../source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# 速查表
[Chisel3 速查表](https://github.com/freechipsproject/chisel-cheatsheet/releases/latest/download/chisel_cheatsheet.pdf) 包含所有主要硬件构造 API 的摘要，包括我们下面将介绍的一些标准库工具。

# Decoupled：标准就绪-有效接口
Chisel 提供的常用接口之一是 `DecoupledIO`，它提供了一个用于传输数据的就绪-有效接口。其思想是：源端在 `bits` 信号上驱动要传输的数据，并在有数据要传输时驱动 `valid` 信号。接收端在准备好接收数据时驱动 `ready` 信号，当 `ready` 和 `valid` 在一个周期内同时被断言时，表示数据传输完成。

这在数据传输的两个方向上提供了一种流控制机制，包括反压机制。

注意：`ready` 和 `valid` 不应组合耦合，否则可能导致无法综合的组合环路。`ready` 应仅取决于接收端是否能够接收数据，`valid` 应仅取决于源端是否有数据。只有在传输完成后（下一个时钟周期），值才应更新。

任何 Chisel 数据都可以包装在 `DecoupledIO` 中（用作 `bits` 字段），如下所示：

```scala
val myChiselData = UInt(8.W)
// or any Chisel data type, such as Bool(), SInt(...), or even custom Bundles
val myDecoupled = Decoupled(myChiselData)
```

上述代码创建了一个新的 `DecoupledIO` Bundle，包含以下字段：
- `valid`: Output(Bool)
- `ready`: Input(Bool)
- `bits`: Output(UInt(8.W))
___

本节其余部分的结构将与之前略有不同：我们不会给你编码练习，而是提供一些代码示例和测试用例，它们会打印电路状态。在运行测试之前，尝试预测会打印什么。

## 队列

`Queue` 创建一个 FIFO（先进先出）队列，两端都有 Decoupled 接口，支持反压。数据类型和元素数量都是可配置的。

>注意，前几次我们打印 `peek` 到的值时，我们对其调用了 `litValue`。这将返回的 chisel 字面量转换为 `BigInt`。后面的类似调用我们没有调用 `litValue`，你可以看到 `peek` 返回的值的额外信息，比如类型和位宽。

In [ ]:
test(new Module {
    // Example circuit using a Queue
    val io = IO(new Bundle {
      val in = Flipped(Decoupled(UInt(8.W)))
      val out = Decoupled(UInt(8.W))
    })
    val queue = Queue(io.in, 2)  // 2-element queue
    io.out <> queue
  }) { c =>
    c.io.out.ready.poke(false.B)
    c.io.in.valid.poke(true.B)  // Enqueue an element
    c.io.in.bits.poke(42.U)
    println(s"Starting:")
    println(s"\tio.in: ready=${c.io.in.ready.peek().litValue}")
    println(s"\tio.out: valid=${c.io.out.valid.peek().litValue}, bits=${c.io.out.bits.peek().litValue}")
    c.clock.step(1)

    c.io.in.valid.poke(true.B)  // Enqueue another element
    c.io.in.bits.poke(43.U)
    // What do you think io.out.valid and io.out.bits will be?
    println(s"After first enqueue:")
    println(s"\tio.in: ready=${c.io.in.ready.peek().litValue}")
    println(s"\tio.out: valid=${c.io.out.valid.peek().litValue}, bits=${c.io.out.bits.peek().litValue}")
    c.clock.step(1)

    c.io.in.valid.poke(true.B)  // Read a element, attempt to enqueue
    c.io.in.bits.poke(44.U)
    c.io.out.ready.poke(true.B)
    // What do you think io.in.ready will be, and will this enqueue succeed, and what will be read?
    println(s"On first read:")
    println(s"\tio.in: ready=${c.io.in.ready.peek().litValue}")
    println(s"\tio.out: valid=${c.io.out.valid.peek().litValue}, bits=${c.io.out.bits.peek().litValue}")
    c.clock.step(1)

    c.io.in.valid.poke(false.B)  // Read elements out
    c.io.out.ready.poke(true.B)
    // What do you think will be read here?
    println(s"On second read:")
    println(s"\tio.in: ready=${c.io.in.ready.peek().litValue}")
    println(s"\tio.out: valid=${c.io.out.valid.peek().litValue}, bits=${c.io.out.bits.peek().litValue}")
    c.clock.step(1)

    // Will a third read produce anything?
    println(s"On third read:")
    println(s"\tio.in: ready=${c.io.in.ready.peek().litValue}")
    println(s"\tio.out: valid=${c.io.out.valid.peek().litValue}, bits=${c.io.out.bits.peek().litValue}")
    c.clock.step(1)
}

## 仲裁器
仲裁器将数据从 _n_ 个 `DecoupledIO` 源端路由到一个 `DecoupledIO` 接收端，基于优先级。
Chisel 中包含两种类型：
- `Arbiter`：优先选择索引较低的生产者
- `RRArbiter`：以轮询顺序运行

注意仲裁器路由是通过组合逻辑实现的。

下面的示例演示了优先级仲裁器的使用（你也将在下一节中实现它）：

In [ ]:
test(new Module {
    // Example circuit using a priority arbiter
    val io = IO(new Bundle {
      val in = Flipped(Vec(2, Decoupled(UInt(8.W))))
      val out = Decoupled(UInt(8.W))
    })
    // Arbiter doesn't have a convenience constructor, so it's built like any Module
    val arbiter = Module(new Arbiter(UInt(8.W), 2))  // 2 to 1 Priority Arbiter
    arbiter.io.in <> io.in
    io.out <> arbiter.io.out
  }) { c =>
    c.io.in(0).valid.poke(false.B)
    c.io.in(1).valid.poke(false.B)
    c.io.out.ready.poke(false.B)
    println(s"Start:")
    println(s"\tin(0).ready=${c.io.in(0).ready.peek().litValue}, in(1).ready=${c.io.in(1).ready.peek().litValue}")
    println(s"\tout.valid=${c.io.out.valid.peek().litValue}, out.bits=${c.io.out.bits.peek().litValue}")
    c.io.in(1).valid.poke(true.B)  // Valid input 1
    c.io.in(1).bits.poke(42.U)
    c.io.out.ready.poke(true.B)
    // What do you think the output will be?
    println(s"valid input 1:")
    println(s"\tin(0).ready=${c.io.in(0).ready.peek().litValue}, in(1).ready=${c.io.in(1).ready.peek().litValue}")
    println(s"\tout.valid=${c.io.out.valid.peek().litValue}, out.bits=${c.io.out.bits.peek().litValue}")
    c.io.in(0).valid.poke(true.B)  // Valid inputs 0 and 1
    c.io.in(0).bits.poke(43.U)
    // What do you think the output will be? Which inputs will be ready?
    println(s"valid inputs 0 and 1:")
    println(s"\tin(0).ready=${c.io.in(0).ready.peek().litValue}, in(1).ready=${c.io.in(1).ready.peek().litValue}")
    println(s"\tout.valid=${c.io.out.valid.peek().litValue}, out.bits=${c.io.out.bits.peek().litValue}")
    c.io.in(1).valid.poke(false.B)  // Valid input 0
    // What do you think the output will be?
    println(s"valid input 0:")
    println(s"\tin(0).ready=${c.io.in(0).ready.peek().litValue}, in(1).ready=${c.io.in(1).ready.peek().litValue}")
    println(s"\tout.valid=${c.io.out.valid.peek().litValue}, out.bits=${c.io.out.bits.peek().litValue}")
}

# 杂项功能块
Chisel Utils 提供了一些执行无状态函数的辅助工具。

## 位运算工具
### PopCount
PopCount 返回输入中高电平（1）位的数量，结果为 `UInt`。

### Reverse
Reverse 返回位反转后的输入。

In [ ]:
test(new Module {
    // Example circuit using PopCount
    val io = IO(new Bundle {
      val in = Input(UInt(8.W))
      val out = Output(UInt(8.W))
    })
    io.out := PopCount(io.in)
  }) { c =>
    // Integer.parseInt is used create an Integer from a binary specification
    c.io.in.poke(Integer.parseInt("00000000", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

    c.io.in.poke(Integer.parseInt("00001111", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

    c.io.in.poke(Integer.parseInt("11001010", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

    c.io.in.poke(Integer.parseInt("11111111", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

}

In [ ]:
test(new Module {
    // Example circuit using Reverse
    val io = IO(new Bundle {
      val in = Input(UInt(8.W))
      val out = Output(UInt(8.W))
    })
    io.out := Reverse(io.in)
  }) { c =>
    // Integer.parseInt is used create an Integer from a binary specification
    c.io.in.poke(Integer.parseInt("01010101", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")

    c.io.in.poke(Integer.parseInt("00001111", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")

    c.io.in.poke(Integer.parseInt("11110000", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")

    c.io.in.poke(Integer.parseInt("11001010", 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")
}

## 独热编码工具
独热码是一种整数编码方式，其中每个值对应一根线，且恰好只有一根线为高电平。这允许高效地创建某些功能，例如多路选择器。然而，如果单线高电平条件不满足，行为可能是未定义的。

以下两个函数提供二进制（`UInt`）和独热码之间的转换，并且互为逆函数：
- UInt 转独热码：`UIntToOH`
- 独热码转 UInt：`OHToUInt`

In [ ]:
test(new Module {
    // Example circuit using UIntToOH
    val io = IO(new Bundle {
      val in = Input(UInt(4.W))
      val out = Output(UInt(16.W))
    })
    io.out := UIntToOH(io.in)
  }) { c =>
    c.io.in.poke(0.U)
    println(s"in=${c.io.in.peek().litValue}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")

    c.io.in.poke(1.U)
    println(s"in=${c.io.in.peek().litValue}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")

    c.io.in.poke(8.U)
    println(s"in=${c.io.in.peek().litValue}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")

    c.io.in.poke(15.U)
    println(s"in=${c.io.in.peek().litValue}, out=0b${c.io.out.peek().litValue.toInt.toBinaryString}")
}

In [ ]:
test(new Module {
    // Example circuit using OHToUInt
    val io = IO(new Bundle {
      val in = Input(UInt(16.W))
      val out = Output(UInt(4.W))
    })
    io.out := OHToUInt(io.in)
}) { c =>
    c.io.in.poke(Integer.parseInt("0000 0000 0000 0001".replace(" ", ""), 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

    c.io.in.poke(Integer.parseInt("0000 0000 1000 0000".replace(" ", ""), 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

    c.io.in.poke(Integer.parseInt("1000 0000 0000 0001".replace(" ", ""), 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

    // Some invalid inputs:
    // None high
    c.io.in.poke(Integer.parseInt("0000 0000 0000 0000".replace(" ", ""), 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")

    // Multiple high
    c.io.in.poke(Integer.parseInt("0001 0100 0010 0000".replace(" ", ""), 2).U)
    println(s"in=0b${c.io.in.peek().litValue.toInt.toBinaryString}, out=${c.io.out.peek().litValue}")
}


## 多路选择器
这些多路选择器接收一个包含值及其选择信号的列表，并输出与最低索引选择信号关联的值。

它们既可以接受一个 (select: Bool, value: Data) 元组的列表，也可以接受对应的选择信号列表和值列表作为参数。为简单起见，下面的示例仅演示第二种形式。

### 优先级多路选择器
`PriorityMux` 输出与最低索引的断言选择信号关联的值。

### 独热多路选择器
`Mux1H` 在保证恰好只有一个选择信号为高电平时提供高效实现。如果该假设不成立，行为是未定义的。

In [ ]:
test(new Module {
    // Example circuit using PriorityMux
    val io = IO(new Bundle {
      val in_sels = Input(Vec(2, Bool()))
      val in_bits = Input(Vec(2, UInt(8.W)))
      val out = Output(UInt(8.W))
    })
    io.out := PriorityMux(io.in_sels, io.in_bits)
  }) { c =>
    c.io.in_bits(0).poke(10.U)
    c.io.in_bits(1).poke(20.U)

    // Select higher index only
    c.io.in_sels(0).poke(false.B)
    c.io.in_sels(1).poke(true.B)
    println(s"in_sels=${c.io.in_sels(0).peek().litValue}, out=${c.io.out.peek().litValue}")

    // Select both - arbitration needed
    c.io.in_sels(0).poke(true.B)
    c.io.in_sels(1).poke(true.B)
    println(s"in_sels=${c.io.in_sels(0).peek().litValue}, out=${c.io.out.peek().litValue}")

    // Select lower index only
    c.io.in_sels(0).poke(true.B)
    c.io.in_sels(1).poke(false.B)
    println(s"in_sels=${c.io.in_sels(0).peek().litValue}, out=${c.io.out.peek().litValue}")
}

In [ ]:
test(new Module {
    // Example circuit using Mux1H
    val io = IO(new Bundle {
      val in_sels = Input(Vec(2, Bool()))
      val in_bits = Input(Vec(2, UInt(8.W)))
      val out = Output(UInt(8.W))
    })
    io.out := Mux1H(io.in_sels, io.in_bits)
  }) { c =>
    c.io.in_bits(0).poke(10.U)
    c.io.in_bits(1).poke(20.U)

    // Select index 1
    c.io.in_sels(0).poke(false.B)
    c.io.in_sels(1).poke(true.B)
    println(s"in_sels=${c.io.in_sels(0).peek().litValue}, out=${c.io.out.peek().litValue}")

    // Select index 0
    c.io.in_sels(0).poke(true.B)
    c.io.in_sels(1).poke(false.B)
    println(s"in_sels=${c.io.in_sels(0).peek().litValue}, out=${c.io.out.peek().litValue}")

    // Select none (invalid)
    c.io.in_sels(0).poke(false.B)
    c.io.in_sels(1).poke(false.B)
    println(s"in_sels=${c.io.in_sels(0).peek().litValue}, out=${c.io.out.peek().litValue}")

    // Select both (invalid)
    c.io.in_sels(0).poke(true.B)
    c.io.in_sels(1).poke(true.B)
    println(s"in_sels=${c.io.in_sels(0).peek().litValue}, out=${c.io.out.peek().litValue}")
}

## 计数器
`Counter` 是一个计数器，每个周期可以递增一次，直到达到某个指定的上限，此时它会发生溢出。注意它**不是**一个模块，它的值是可访问的。

In [ ]:
test(new Module {
    // Example circuit using Mux1H
    val io = IO(new Bundle {
      val count = Input(Bool())
      val out = Output(UInt(2.W))
    })
    val counter = Counter(3)  // 3-count Counter (outputs range [0...2])
    when(io.count) {
      counter.inc()
    }
    io.out := counter.value
  }) { c =>
    c.io.count.poke(true.B)
    println(s"start: counter value=${c.io.out.peek().litValue}")

    c.clock.step(1)
    println(s"step 1: counter value=${c.io.out.peek().litValue}")

    c.clock.step(1)
    println(s"step 2: counter value=${c.io.out.peek().litValue}")

    c.io.count.poke(false.B)
    c.clock.step(1)
    println(s"step without increment: counter value=${c.io.out.peek().litValue}")

    c.io.count.poke(true.B)
    c.clock.step(1)
    println(s"step again: counter value=${c.io.out.peek().litValue}")
}

---
# 你已完成！

[返回顶部](#top)